### Preparing k-fold train-test/val splits

#### Create a csv file with slide_name and label from the WSI folders.

In [1]:
#!/usr/bin/env python3
"""Create slide CSV with custom Normal=0, Tumor=1 mapping for the dataset with an example CAMELYON16."""
import pandas as pd
from pathlib import Path
import sys

# Configuration
dataset = 'CAMELYON16'
path = '/data_64T_3/Dataset/CAMELYON16/images'
extensions = ['*.svs', '*.tif', '*.tiff']
tissues = ['Normal', 'Tumor']
output = f'/data_64T_3/Dataset/CAMELYON16/labels/{dataset}_binary_class_label.csv'

# Exclusion list - add WSI names (without extension) to exclude
exclude_list = [
    # 'slide_name_1',
    # 'slide_name_2',
    # Add more slide names here as needed
]

# Custom label mapping
label_map = {tissues[0] : 0, tissues[1] : 1}

# Generate data
data = []
for tissue in tissues:
    tissue_folder = Path(path) / tissue
    if not tissue_folder.exists():
        print(f"⚠️  Warning: {tissue_folder} does not exist, skipping...")
        continue
    
    # Check if it's a directory structure or files directly in tissue folder
    svs_files = [f for ext in extensions for f in tissue_folder.glob(ext)]
    
    # print(f"tissue_folder: {tissue_folder}")
    # print(f"len(svs_files) : {len(svs_files)}")

    if svs_files:
        # Files are directly in the tissue folder
        label = label_map[tissue]
        for image in sorted(svs_files):
            if image.stem not in exclude_list:
                data.append({'slide_name': image.stem, 'label': label})

# # Save CSV
df = pd.DataFrame(data)
df.to_csv(output, index=False)

print(f"✓ Created {output}")
print(f"  Total slides: {len(df)}")
print(f"  Excluded slides: {len(exclude_list)}")
print(f"  Normal (0): {(df['label']==0).sum()}")
print(f"  Tumor (1): {(df['label']==1).sum()}")

✓ Created /data_64T_3/Dataset/CAMELYON16/labels/CAMELYON16_binary_class_label.csv
  Total slides: 399
  Excluded slides: 0
  Normal (0): 239
  Tumor (1): 160


#### Creating csv files with feature path

In [4]:
#!/usr/bin/env python3
"""Add feature paths to WSI names in CSV files."""
import pandas as pd
from pathlib import Path
import sys

base_path = "/data_64T_3/Shared/Extracted_Features"
dataset =  "CAMELYON16"   # TCGA_LUAD
pre_processor = "MUFASA"    # MUFASA, TRIDENT, CLAM, HistoLab 
tissue = ["Normal", "Tumor"]
encoder = "resnet50_1024"   # 'resnet18', 'uni'

# Configuration
csv_dir = '/data_64T_3/Dataset/CAMELYON16/labels/New folder'
feature_dirs = [f'{base_path}/{pre_processor}/{dataset}/{tissue[0]}/Extracted_features/{encoder}',
               f'{base_path}/{pre_processor}/{dataset}/{tissue[1]}/Extracted_features/{encoder}']

outut_path = "/Project/7.Multi_Phase_Tile_Extractor_Experiments/WSI_Classification/MIL_BASELINE/datasets/"

# Build feature lookup dictionary
print("Building feature file index...")
feature_map = {}
for feature_dir in feature_dirs:
    for pt_file in Path(feature_dir).rglob('*.pt'):
        slide_name = pt_file.stem
        feature_map[slide_name] = str(pt_file)
print(f"Found {len(feature_map)} feature files across {len(feature_dirs)} directories")

# Process all CSV files
for csv_file in Path(csv_dir).glob('*.csv'):
    df = pd.read_csv(csv_file)
    missing = []
    
    # Update all *_slide_path columns
    for col in df.columns:
        if col.endswith('_slide_path'):
            def get_path(x):
                if pd.notna(x):
                    if x in feature_map:
                        return feature_map[x]
                    else:
                        missing.append(x)
                        return x  # Keep original if not found
                return x
            df[col] = df[col].apply(get_path)
    
    # Save with _with_path suffix
    output_file = csv_file.parent / f"{csv_file.stem}_with_path.csv"
    df.to_csv(output_file, index=False)
    print(f"✓ {csv_file.name} → {output_file.name}")
    if missing:
        print(f"  ⚠️  Missing {len(set(missing))} features: {set(missing)}")

Building feature file index...
Found 399 feature files across 2 directories
✓ Camelyon16_binary_class_label_common_split.csv → Camelyon16_binary_class_label_common_split_with_path.csv


##### TCGA_NSCLC (LUSC vs LUAD) - Preparing based on different folders

In [5]:
#!/usr/bin/env python3
"""Create slide CSV combining multiple specific directory paths with custom labels."""
import pandas as pd
from pathlib import Path

# --- Configuration ---
output_path = '/data_64T_2/Dataset/TCGA_LUAD/labels/TCGA_LUAD_vs_LUSC_binary.csv'
extensions = ['*.svs', '*.tif', '*.tiff']
exclude_list = []

# Define your sources: (Path, Label)
sources = [
    ("/data_64T_2/Dataset/TCGA_LUAD/images/Tumor_DX1", 0),
    ("/data_64T_3/Dataset/TCGA_LUSC/images/Tumor_DX1", 1)
]

# --- Processing ---
data = []

for folder_path, label in sources:
    folder = Path(folder_path)
    
    if not folder.exists():
        print(f"⚠️  Warning: {folder} does not exist, skipping...")
        continue

    # 1. Gather files with multiple extensions
    # 2. Filter exclusions
    # 3. Create dictionary entry
    files = [
        {'slide_name': f.stem, 'label': label}
        for ext in extensions
        for f in folder.glob(ext)
        if f.stem not in exclude_list
    ]
    
    data.extend(files)
    print(f"-> Found {len(files)} slides in {folder.name} (Label {label})")

# --- Save & Summary ---
if data:
    df = pd.DataFrame(data)
    # Optional: Sort by slide name for consistency
    df = df.sort_values(by='slide_name')
    
    # Create parent directory if it doesn't exist
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)

    print(f"\n✓ Created {output_path}")
    print(f"  Total slides: {len(df)}")
    print(f"  Label 0 (LUAD): {(df['label'] == 0).sum()}")
    print(f"  Label 1 (LUSC): {(df['label'] == 1).sum()}")
else:
    print("\n❌ No slides found in any specified directory.")

-> Found 398 slides in Tumor_DX1 (Label 0)
-> Found 477 slides in Tumor_DX1 (Label 1)

✓ Created /data_64T_2/Dataset/TCGA_LUAD/labels/TCGA_LUAD_vs_LUSC_binary.csv
  Total slides: 875
  Label 0 (LUAD): 398
  Label 1 (LUSC): 477


##### TCGA_NSCLC (LUSC vs LUAD) - Preparing feature paths based on different csv files

In [8]:
#!/usr/bin/env python3
"""Add feature paths to WSI names in CSV files."""
import pandas as pd
from pathlib import Path
import sys

base_path = "/data_64T_3/Shared/Extracted_Features"  
dataset = "TCGA_NSCLC"    # TCGA_LUAD
pre_processor = "MUFASA"   # CLAM, HISTOLAB, MUFASA, TRIDENT
encoder = "resnet50_1024"   # 'resnet18', 'uni'

# Configuration
csv_dir = f'/data_64T_2/Dataset/TCGA_NSCLC/labels/5-fold-common-split' # path that contains split inforamation. 
# Even if there is a single csv file, place it in a folder and provide its location

# feature location
feature_dirs = [f'/data_64T_3/Shared/Extracted_Features/{pre_processor}/TCGA_LUSC/Tumor_DX1/Extracted_features/resnet50_1024',
                f'/data_64T_3/Shared/Extracted_Features/{pre_processor}/TCGA_LUAD/Tumor_DX1/Extracted_features/resnet50_1024']

# output path to store
output_path = f"/data_64T_3/Shared/Extracted_Features/{pre_processor}/TCGA_NSCLC/{dataset}_{pre_processor}_{encoder}_splits"

Path(output_path).mkdir(parents=True, exist_ok=True)
# ------------------------------------------------------------------------------------------------
# Build feature lookup dictionary
print("Building feature file index...")
feature_map = {}
dirs_found = True # Flag to track if all feature directories exist

for feature_dir in feature_dirs:
    # Check if directory exists to avoid errors
    if Path(feature_dir).exists():
        for pt_file in Path(feature_dir).rglob('*.pt'):
            slide_name = pt_file.stem
            feature_map[slide_name] = str(pt_file)
    else:
        print(f"Warning: Directory not found: {feature_dir}")
        dirs_found = False # Set flag to False if any directory is missing

if not dirs_found:
    print("⚠️  Stopping: One or more feature directories are missing.")
    sys.exit(1) # Exit script if directories are bad

print(f"Found {len(feature_map)} feature files across {len(feature_dirs)} directories")

# Process all CSV files
for csv_file in Path(csv_dir).glob('*.csv'):
    df = pd.read_csv(csv_file)
    missing = []
    
    # Update all *_slide_path columns
    for col in df.columns:
        if col.endswith('_slide_path'):
            def get_path(x):
                if pd.notna(x):
                    # Ensure x is a string to match dictionary keys safely
                    x_str = str(x).strip() 
                    if x_str in feature_map:
                        return feature_map[x_str]
                    else:
                        missing.append(x_str)
                        return x  # Keep original if not found
                return x
            
            df[col] = df[col].apply(get_path)
    
    # --- CHANGED LOGIC HERE ---
    if missing:
        unique_missing = set(missing)
        print(f"⚠️  SKIPPING {csv_file.name}: Missing {len(unique_missing)} features: {unique_missing}")
        # Do NOT save the file, simply continue to the next one

    # Only reach here if missing list is empty
    output_file = f'{output_path}/{csv_file.stem}_with_path.csv'
    df.to_csv(output_file, index=False)
    
    print(f"✓ {csv_file.name} -> {Path(output_file).name}")

Building feature file index...
Found 869 feature files across 2 directories
✓ Total_5-fold_TCGA_NSCLC_5fold.csv -> Total_5-fold_TCGA_NSCLC_5fold_with_path.csv
✓ Total_5-fold_TCGA_NSCLC_1fold.csv -> Total_5-fold_TCGA_NSCLC_1fold_with_path.csv
✓ Total_5-fold_TCGA_NSCLC_3fold.csv -> Total_5-fold_TCGA_NSCLC_3fold_with_path.csv
✓ Total_5-fold_TCGA_NSCLC_4fold.csv -> Total_5-fold_TCGA_NSCLC_4fold_with_path.csv
✓ Total_5-fold_TCGA_NSCLC_2fold.csv -> Total_5-fold_TCGA_NSCLC_2fold_with_path.csv


#### For multiple labels

In [1]:
import pandas as pd

# Configuration
input_csv = "/data_64T_3/Dataset/CAMELYON17/labels/Camelyon+(4-classes).csv"
output_csv = "/data_64T_3/Dataset/CAMELYON17/labels/Multi-class.csv"

# Mapping dictionary
label_map = {'negative': 0, 'macro': 1, 'micro': 2, 'itc': 3}

# Process: Read -> Rename -> Map -> Save
df = pd.read_csv(input_csv)
df = df.rename(columns={'slide': 'slide_name'})
df['label'] = df['label'].map(label_map)

# Save to new file
df.to_csv(output_csv, columns=['slide_name', 'label'], index=False)

print(f"Saved to {output_csv}")

Saved to /data_64T_3/Dataset/CAMELYON17/labels/Multi-class.csv
